# BugBuster — Test Dashboard

**Architecture:** BBP v4 · `bugbuster` Python library · pytest with USB/HTTP/sim transports  
**Last refreshed:** 2026-05-04

Operator-friendly test runner for all BugBuster test layers.

| Section | Test layer | Hardware? |
|---|---|---|
| 1 | Setup + environment check | No |
| 2 | Unit suite (`tests/unit`) | No |
| 3 | Simulator suite (`tests/device --sim`) | No |
| 4 | HTTP API suite (`tests/http_api`) | Optional |
| 5 | USB-only subset | Yes |
| 6 | HAT subset | Yes (HAT attached) |
| 7 | Summary dashboard | — |

**Markers:** `usb_only` · `requires_hat` · `destructive` · `slow` · `streaming`  
**Env vars:** `BUGBUSTER_PORT` · `BUGBUSTER_HTTP_HOST` · `BUGBUSTER_ADMIN_TOKEN`

## 1. Setup + environment check

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Repo root is one level up from Notebooks/
REPO_ROOT = Path(os.getcwd()).parent
TESTS_DIR = REPO_ROOT / 'tests'
VENV_PY   = REPO_ROOT / '.venv' / 'bin' / 'python'

# Prefer venv python if present, fall back to current interpreter
PYTHON = str(VENV_PY) if VENV_PY.exists() else sys.executable

# PYTHONPATH includes both python/ (library) and tests/ (mock/simulator)
PYPATH = os.pathsep.join([
    str(REPO_ROOT / 'python'),
    str(REPO_ROOT / 'tests'),
])

print(f'Repo root  : {REPO_ROOT}')
print(f'Python     : {PYTHON}')
print(f'PYTHONPATH : {PYPATH}')

# Verify bugbuster importable
r = subprocess.run([PYTHON, '-c', 'import bugbuster; print("bugbuster", bugbuster.__version__)'],
                   capture_output=True, text=True,
                   env={**os.environ, 'PYTHONPATH': PYPATH})
print(r.stdout.strip() or r.stderr.strip())

# Verify pytest
r2 = subprocess.run([PYTHON, '-m', 'pytest', '--version'],
                    capture_output=True, text=True)
print(r2.stdout.strip() or r2.stderr.strip())

In [ ]:
def run_pytest(args, label=''):
    """Run pytest with given args, stream output, return return-code."""
    cmd = [PYTHON, '-m', 'pytest'] + args
    env = {**os.environ, 'PYTHONPATH': PYPATH}
    if label:
        print(f'=== {label} ===')
    print(f'$ {" ".join(cmd)}')
    print('-' * 60)
    proc = subprocess.Popen(cmd, cwd=str(REPO_ROOT), env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print('-' * 60)
    print(f'Exit code: {proc.returncode}')
    return proc.returncode

print('run_pytest() helper ready.')

## 2. Unit suite

Pure-Python logic tests — no hardware, no simulator. Always fast.

In [ ]:
run_pytest(
    [str(TESTS_DIR / 'unit'), '-q'],
    label='Unit suite',
)

## 3. Simulator suite

End-to-end tests routed through `SimulatedDevice` — no hardware required.
Covers all BBP v4 command handlers and the HTTP `/api/*` surface.

In [ ]:
run_pytest(
    [str(TESTS_DIR / 'device'), '--sim', '-q'],
    label='Simulator suite (tests/device --sim)',
)

## 4. HTTP API suite

Contract tests against the HTTP REST endpoints.
If `BUGBUSTER_HTTP_HOST` is set, runs against real hardware.
Otherwise runs against the simulator.

In [ ]:
http_host = os.environ.get('BUGBUSTER_HTTP_HOST', None)

if http_host:
    run_pytest(
        [str(TESTS_DIR / 'http_api'), '-q',
         f'--device-http={http_host}'],
        label=f'HTTP API suite ({http_host})',
    )
else:
    run_pytest(
        [str(TESTS_DIR / 'http_api'), '-q', '--sim'],
        label='HTTP API suite (simulator)',
    )

## 5. USB-only subset

Tests marked `usb_only` (streaming, register access, HAT power).
Set `BUGBUSTER_PORT` to your device's serial port.

In [ ]:
usb_port = os.environ.get('BUGBUSTER_PORT', None)

if usb_port:
    run_pytest(
        [str(TESTS_DIR / 'device'), '-q', '-m', 'usb_only',
         f'--device-usb={usb_port}'],
        label=f'USB-only subset ({usb_port})',
    )
else:
    print('BUGBUSTER_PORT not set — skipping USB-only tests.')
    print('Set it with: import os; os.environ["BUGBUSTER_PORT"] = "/dev/cu.usbmodem1234"')

## 6. HAT subset

Tests marked `requires_hat`. Requires USB device + RP2040 HAT attached.

In [ ]:
if usb_port:
    run_pytest(
        [str(TESTS_DIR / 'device'), '-q', '-m', 'requires_hat',
         f'--device-usb={usb_port}', '--hat'],
        label=f'HAT subset ({usb_port})',
    )
else:
    print('BUGBUSTER_PORT not set — skipping HAT tests.')

## 7. Summary dashboard

In [ ]:
import json as _json

REPORT_JSON = TESTS_DIR / '.test_results.json'

# Run unit + simulator suite capturing JSON report
extra = []
try:
    subprocess.run([PYTHON, '-c', 'import pytest_json_report'],
                   capture_output=True, check=True)
    extra = ['--json-report', f'--json-report-file={REPORT_JSON}']
except subprocess.CalledProcessError:
    pass  # json-report not installed

rc = run_pytest(
    [str(TESTS_DIR / 'unit'),
     str(TESTS_DIR / 'device'), '--sim',
     '-q', '--tb=no'] + extra,
    label='Full hardware-free suite',
)

# Parse JSON report if available
if REPORT_JSON.exists():
    with open(REPORT_JSON) as f:
        report = _json.load(f)
    summary  = report.get('summary', {})
    total    = summary.get('total', 0)
    passed   = summary.get('passed', 0)
    failed   = summary.get('failed', 0)
    skipped  = summary.get('skipped', 0)
    duration = report.get('duration', 0)

    print()
    print('=' * 50)
    print('  BugBuster Test Summary (hardware-free)')
    print('=' * 50)
    print(f'  Total   : {total}')
    print(f'  Passed  : {passed}')
    print(f'  Failed  : {failed}')
    print(f'  Skipped : {skipped}')
    print(f'  Duration: {duration:.1f}s')
    print('=' * 50)
    if failed == 0:
        print('  All tests passed.')
    else:
        print(f'  {failed} test(s) FAILED — see output above.')
    if failed > 0:
        print()
        for t in report.get('tests', []):
            if t.get('outcome') == 'failed':
                print(f'  FAIL: {t["nodeid"]}')
else:
    status = 'PASSED' if rc == 0 else 'FAILED'
    print(f'\nSuite result: {status} (exit code {rc})')
    print('Install pytest-json-report for a detailed summary: pip install pytest-json-report')

html_report = TESTS_DIR / 'report.html'
if html_report.exists():
    print(f'\nHTML report: {html_report}')

### Full hardware-in-the-loop run via run_tests.py

```bash
# USB only
python tests/run_tests.py --usb /dev/cu.usbmodem1234

# USB + HTTP + HAT
python tests/run_tests.py --usb /dev/cu.usbmodem1234 --http 192.168.4.1 --hat

# Skip destructive tests
python tests/run_tests.py --usb /dev/cu.usbmodem1234 --skip-destructive

# Single category
python tests/run_tests.py --usb /dev/cu.usbmodem1234 --category channels

# With HTML report
python tests/run_tests.py --usb /dev/cu.usbmodem1234 --html-report
```